# Stage 0 — Lighting-Robustness Stress Test + Boundary IoU

Two evaluation-only experiments on the **frozen baseline checkpoints**. No training.

**A. Lighting stress test** — re-segment + re-evaluate the frozen 4ch/7ch models on
photometrically perturbed copies of the test orthomosaics (brightness / contrast /
gamma / shadow ramp). Normal maps, alpha channels, and GT masks are untouched, so
the 3ch geometry model is invariant *by construction* and serves as the flat
reference. The question this answers is the paper's actual motivation: **does the
fusion model stay stable when appearance degrades, where the appearance-only model
does not?**

**B. Boundary IoU** — quantify the Fig.-7 claim ("the full model produces cleaner
stone boundaries") on the already-saved RAW rasters of all three variants.
CPU-only; needs no GPU at all.

Design notes:
- Each perturbation condition gets its **own experiment dir**
  (`v6_stress_<condition>`) with its own manifest — the Step-4 per-fraction
  pattern, so nothing ever collides or overwrites.
- Checkpoints are **read** from the frozen experiment via the new
  `checkpoint_experiment` config field; the frozen experiment is never written to.
- The `none` condition runs the *original* orthos through the identical harness —
  it must reproduce the frozen baseline numbers, a built-in sanity check.
- Everything is skip-if-exists; an interrupted sweep resumes cleanly.

Workflow: (1) CONFIG → (2) preview & path check (no GPU) → (3) run stress sweep →
(4) degradation curves → (5) boundary eval → (6) boundary summary.

## 0. Imports

In [ ]:
import os, sys, glob, dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import amg_pipeline as amg
from amg_pipeline.config import RunConfig, make_run_id
from amg_pipeline import paths
from amg_pipeline.perturb import condition_name, apply_perturbation
from amg_pipeline.stress import run_stress, collect_stress_manifests
from amg_pipeline.boundary import run_boundary_eval

print("amg_pipeline loaded from:", os.path.dirname(amg.__file__))

## 1. CONFIG — set everything here

The paths mirror the main orchestrator. **Check `FROZEN_EXPERIMENT`**: it must be
the exact folder name (under `experiments/`) that holds the frozen baseline
checkpoints — the preview cell verifies every checkpoint exists before anything runs.

In [ ]:
# === FROZEN BASELINE (checkpoints are READ from here, never written) ===
FROZEN_EXPERIMENT = "v2_yaw_correction-epsV1"   # <-- verify against your experiments/ folder!

# === STAGE-0 EXPERIMENT IDENTITY ===
STRESS_BASE   = "v6_stress"      # each condition becomes v6_stress_<condition>
BOUNDARY_OUT  = "v6_boundary"    # boundary manifest goes to experiments/v6_boundary/
EXPERIMENTS_ROOT = os.path.join(REPO_ROOT, "experiments")

# === STRESS SWEEP SHAPE ===
# 3ch is deliberately absent: its inputs are untouched by every condition, so it is
# invariant by construction; its flat reference comes from the frozen manifest.
CHANNEL_VARIANTS = (4, 7)
N_RUNS = 5

# === PERTURBATION GRID (levels; 1.0 = identity) ===
# Trim levels here if you want a shorter first pass — every condition is
# independent and resume-safe, so you can also add levels later.
LEVELS = {
    "bright":   (0.60, 0.80, 1.20, 1.40),   # multiplicative brightness
    "contrast": (0.60, 0.80, 1.20, 1.40),   # scaled around mean wall luminance
    "gamma":    (0.67, 0.80, 1.25, 1.50),   # x**level
    "shadow":   (0.70, 0.50, 0.30),          # horizontal ramp 1.0 -> level
}
CONDITIONS = [("none", 1.0)] + [(f, l) for f, lvls in LEVELS.items() for l in lvls]

# === TEST WALLS (same as the orchestrator) ===
TEST_ORTHO_DIR     = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\01_test-images"
TEST_NORMALMAP_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\02_test-normals"
TEST_MASK_DIR      = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\03_test-masks"
WALLS = ["wall1", "wall2", "wall3", "wall4"]
ORTHO_PATTERN     = "{wall}_png-ortho.png"
NORMALMAP_PATTERN = "{wall}_DEM_normalmap.png"
MASK_PATTERN      = "{wall}_png-ortho.png"

# === ROI EVAL (identical to the frozen pipeline) ===
ROI_OPERATION = "closing"
KERNEL_RADIUS = 45

# === BOUNDARY IoU ===
BOUNDARY_RADII    = (5, 15)      # band radius in px; two radii show d-robustness
BOUNDARY_VARIANTS = (3, 4, 7)    # boundary eval includes 3ch (its rasters exist)

BASE_CONFIG = RunConfig(
    channels=7, run_number=1,
    experiment_name=STRESS_BASE, experiments_root=EXPERIMENTS_ROOT,
    test_ortho_dir=TEST_ORTHO_DIR, test_normalmap_dir=TEST_NORMALMAP_DIR,
    test_mask_dir=TEST_MASK_DIR, walls=WALLS,
    ortho_pattern=ORTHO_PATTERN, normalmap_pattern=NORMALMAP_PATTERN, mask_pattern=MASK_PATTERN,
    roi_operation=ROI_OPERATION, kernel_radius=KERNEL_RADIUS,
)
print(f"Config OK. {len(CONDITIONS)} conditions x {len(CHANNEL_VARIANTS)} variants x {N_RUNS} runs "
      f"= {len(CONDITIONS)*len(CHANNEL_VARIANTS)*N_RUNS} segment+evaluate passes (no training).")

## 2. Preview & check paths — NO GPU work

Verifies every frozen checkpoint and every test input exists, lists the derived
experiment names, and (optionally) renders a downscaled visual preview of each
perturbation family at its strongest level. **Fix any ✗ before running the sweep.**

In [ ]:
DO_PREVIEW_IMAGE = True   # render perturbation preview on a downscaled wall1

print("=== FROZEN CHECKPOINTS ===")
all_ok = True
for ch in sorted(set(CHANNEL_VARIANTS) | set(BOUNDARY_VARIANTS)):
    for run_n in range(1, N_RUNS + 1):
        cfg = dataclasses.replace(BASE_CONFIG, channels=ch, run_number=run_n,
                                  experiment_name=FROZEN_EXPERIMENT)
        p = paths.checkpoint_path(cfg)
        ok = os.path.exists(p)
        all_ok &= ok
        if not ok:
            print(f"  X   {make_run_id(cfg)}  MISSING: {p}")
print("  all present" if all_ok else "  *** fix FROZEN_EXPERIMENT or missing runs ***")

print("\n=== TEST WALL INPUTS ===")
for wall in WALLS:
    for label, fn in [("ortho", paths.test_ortho_path(BASE_CONFIG, wall)),
                      ("mask",  paths.test_mask_path(BASE_CONFIG, wall))]:
        exists = os.path.exists(fn)
        all_ok &= exists
        print(f"  {'OK ' if exists else 'X  '} {wall} {label:6s} {fn}")

print("\n=== CONDITIONS -> EXPERIMENT DIRS ===")
for fam, lvl in CONDITIONS:
    cond = condition_name(fam, lvl)
    src = "original test orthos" if fam == "none" else f"{STRESS_BASE}_inputs/{cond}"
    print(f"  {cond:12s} -> {STRESS_BASE}_{cond}   (orthos: {src})")

print("\nAll inputs present." if all_ok else "\n*** Some inputs MISSING — fix before running. ***")

if DO_PREVIEW_IMAGE:
    import cv2
    src = paths.test_ortho_path(BASE_CONFIG, WALLS[0])
    img = cv2.imread(src, cv2.IMREAD_UNCHANGED)
    scale = 900 / img.shape[1]
    small = cv2.resize(img, (900, int(img.shape[0] * scale)), interpolation=cv2.INTER_AREA)
    color = cv2.cvtColor(small[:, :, :3], cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    mask = small[:, :, 3] > 0 if small.shape[2] == 4 else None
    strongest = [("none", 1.0)] + [(f, lvls[np.argmax(np.abs(np.array(lvls) - 1.0))])
                                   for f, lvls in LEVELS.items()]
    fig, axes = plt.subplots(1, len(strongest), figsize=(3.4 * len(strongest), 3.2))
    for ax, (fam, lvl) in zip(axes, strongest):
        ax.imshow(apply_perturbation(color, fam, lvl, mask))
        ax.set_title(condition_name(fam, lvl), fontsize=10); ax.axis("off")
    plt.suptitle(f"{WALLS[0]}: strongest level per family (preview, downscaled)")
    plt.tight_layout(); plt.show()

## 3. Run the stress sweep

Gated by `DO_RUN`. For each condition: write perturbed orthos (skip-if-exists),
then segment + evaluate every (variant, run) with the **frozen** checkpoints and
append to that condition's manifest. Resume-safe throughout.

Disk note: perturbed ortho folders total ~(#non-identity conditions) x (size of the
test-ortho folder). They are deterministic and regenerable — you can delete
`experiments/v6_stress_inputs/` after the sweep if space is tight.

In [ ]:
DO_RUN = False        # <-- set True to launch
FORCE  = False

if DO_RUN:
    run_stress(BASE_CONFIG, FROZEN_EXPERIMENT, STRESS_BASE, CONDITIONS,
               channel_variants=CHANNEL_VARIANTS, n_runs=N_RUNS, force=FORCE)
else:
    total = len(CONDITIONS) * len(CHANNEL_VARIANTS) * N_RUNS
    print(f"DO_RUN is False; set it to True to launch {total} segment+evaluate passes.")

## 4. Degradation curves

One panel per perturbation family: AllWalls metric vs. level (1.0 = original),
mean ± std over the `N_RUNS` repeats. The dashed blue line is the 3ch geometry
model from the frozen manifest — flat by construction. The `none` condition is
plotted at level 1.0 in every panel and printed against the frozen 4ch/7ch numbers
as the harness sanity check (they should agree to ~1e-3).

In [ ]:
METRIC = "IoU_mean_stones"   # try "IoU_Quarry" for the class where fusion leads

df = collect_stress_manifests(BASE_CONFIG, STRESS_BASE, CONDITIONS,
                              frozen_experiment=FROZEN_EXPERIMENT)
aw = df[df.wall == "AllWalls"].copy()

# --- harness sanity: 'none' must reproduce the frozen numbers -----------------
print("=== SANITY: none vs frozen (AllWalls mean over runs) ===")
for ch in CHANNEL_VARIANTS:
    a = aw[(aw.condition == "none") & (aw.channels == ch)][METRIC].mean()
    b = aw[(aw.condition == "frozen") & (aw.channels == ch)][METRIC].mean()
    print(f"  {ch}ch  none={a:.4f}  frozen={b:.4f}  delta={a-b:+.4f}")

ref3 = aw[(aw.condition == "frozen") & (aw.channels == 3)][METRIC]

families = list(LEVELS.keys())
fig, axes = plt.subplots(1, len(families), figsize=(4.3 * len(families), 3.8), sharey=True)
none_rows = aw[aw.condition == "none"].copy(); none_rows["level"] = 1.0
styles = {4: ("#eda100", "appearance (4ch)"), 7: ("#1baf7a", "full fusion (7ch)")}
for ax, fam in zip(np.atleast_1d(axes), families):
    sub = pd.concat([aw[aw.family == fam], none_rows], ignore_index=True)
    for ch, (color, label) in styles.items():
        g = (sub[sub.channels == ch].groupby("level")[METRIC]
             .agg(["mean", "std"]).sort_index())
        ax.errorbar(g.index, g["mean"], yerr=g["std"], marker="o", capsize=3,
                    lw=2, color=color, label=label)
    if len(ref3):
        ax.axhline(ref3.mean(), ls="--", lw=1.5, color="#2a78d6", label="geometry (3ch, invariant)")
        ax.axhspan(ref3.mean() - ref3.std(), ref3.mean() + ref3.std(),
                   color="#2a78d6", alpha=0.10)
    ax.axvline(1.0, color="gray", lw=0.8, ls=":")
    ax.set_title(fam); ax.set_xlabel("level (1.0 = original)")
    ax.grid(alpha=0.25)
np.atleast_1d(axes)[0].set_ylabel(METRIC)
np.atleast_1d(axes)[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

# --- compact table -------------------------------------------------------------
tbl = (aw[aw.condition != "frozen"]
       .pivot_table(index="condition", columns="channels", values=METRIC, aggfunc="mean")
       .round(4))
tbl["gap_7ch_minus_4ch"] = (tbl[7] - tbl[4]).round(4)
print(f"\n=== {METRIC} by condition (mean over runs) ===")
display(tbl.sort_index())

## 5. Boundary IoU on the frozen rasters — CPU only

Quantifies the Fig.-7 boundary-quality claim on the saved RAW segmentations of all
three variants. Bands are computed on the full masks, then restricted to the same
MC-ROI as the pixel metrics; absent classes are dropped (NaN) exactly as in
`evaluate.py`. Two radii show whether the ranking is robust to the band width.

In [ ]:
DO_RUN_BOUNDARY = False   # <-- set True to compute (CPU-only, no GPU needed)
FORCE_BOUNDARY  = False

if DO_RUN_BOUNDARY:
    bdf = run_boundary_eval(BASE_CONFIG, FROZEN_EXPERIMENT, out_name=BOUNDARY_OUT,
                            channel_variants=BOUNDARY_VARIANTS, n_runs=N_RUNS,
                            radii=BOUNDARY_RADII, force=FORCE_BOUNDARY)
else:
    print("DO_RUN_BOUNDARY is False; set it to True to compute Boundary IoU.")

## 6. Boundary IoU summary

In [ ]:
bpath = os.path.join(EXPERIMENTS_ROOT, BOUNDARY_OUT, "boundary_manifest.csv")
if not os.path.exists(bpath):
    print("No boundary manifest yet — run cell 5 first:", bpath)
else:
    bdf = pd.read_csv(bpath)
    baw = bdf[bdf.wall == "AllWalls"]
    for r in sorted(baw.radius_px.unique()):
        print(f"=== Boundary IoU, band radius d = {r}px (AllWalls, mean ± std over runs) ===")
        for ch in sorted(baw.channels.unique()):
            s = baw[(baw.radius_px == r) & (baw.channels == ch)]["BIoU_mean_stones"]
            print(f"  {ch}ch  mean-stone BIoU = {s.mean():.4f} ± {s.std():.4f}")
        print()
    print("=== per-class (AllWalls, mean over runs) ===")
    display(baw.pivot_table(index=["radius_px", "channels"],
                            values=["BIoU_Ashlar", "BIoU_Polygonal", "BIoU_Quarry",
                                    "BIoU_mean_stones"], aggfunc="mean").round(4))
    print("=== per-wall mean-stone BIoU (mean over runs) ===")
    display(bdf[bdf.wall != "AllWalls"]
            .pivot_table(index=["radius_px", "wall"], columns="channels",
                         values="BIoU_mean_stones", aggfunc="mean").round(4))